# Task 3 — K-Means Customer Segmentation (“Whale Hunter”)
We create customer-level RFM features (Recency, Frequency, Monetary), choose `k` using silhouette score, profile clusters, identify the highest-spending “whale” cluster, and generate a marketing brief.

In [ ]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
ART = Path("artifacts"); FIG = Path("figures"); EX = Path("examples")
for p in [ART, FIG, EX]:
    p.mkdir(exist_ok=True)
print("Folders ready:", ART, FIG, EX)

Folders ready: artifacts figures examples


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/aakashsyadav1999/online-retail-dataset/main/online%2Bretail.zip"
LOCAL_FALLBACK = Path("Online Retail.xlsx")

def load_online_retail() -> pd.DataFrame:
    try:
        import urllib.request
        with urllib.request.urlopen(DATA_URL, timeout=60) as resp:
            content = resp.read()
        with zipfile.ZipFile(io.BytesIO(content)) as z:
            xlsx_name = [n for n in z.namelist() if n.lower().endswith(".xlsx")][0]
            with z.open(xlsx_name) as f:
                df = pd.read_excel(f, engine="openpyxl")
        print(f"Loaded dataset from GitHub mirror: {len(df)} rows")
        return df
    except Exception as e:
        print(f"Could not fetch from URL ({e}); trying local fallback '{LOCAL_FALLBACK}'...")
        if LOCAL_FALLBACK.exists():
            df = pd.read_excel(LOCAL_FALLBACK, engine="openpyxl")
            print(f"Loaded dataset from local file: {len(df)} rows")
            return df
        raise RuntimeError(
            "Dataset could not be loaded from the URL or a local file. "
            "Download 'Online Retail.xlsx' (UCI dataset id 352) manually and place it "
            f"next to this script as '{LOCAL_FALLBACK}'."
        )

raw = load_online_retail()
print(raw.head())
print(raw.columns.tolist())

Loaded dataset from GitHub mirror: 541909 rows
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [ ]:
df = raw.copy()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df = df.dropna(subset=["CustomerID"])
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df["Sales"] = df["Quantity"] * df["UnitPrice"]

snapshot = df["InvoiceDate"].max() + pd.Timedelta(days=1)
rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Sales", "sum"),
).reset_index()
rfm.to_csv(EX / "cleaned_dataset_sample.csv", index=False)
print(rfm.head())

   CustomerID  Recency  Frequency  Monetary
0     12346.0      326          1  77183.60
1     12347.0        2          7   4310.00
2     12348.0       75          4   1797.24
3     12349.0       19          1   1757.55
4     12350.0      310          1    334.40


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

features = ["Recency", "Frequency", "Monetary"]
Xs = StandardScaler().fit_transform(np.log1p(rfm[features]))

scores = {}
for k in range(2, 9):
    lab = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(Xs)
    scores[k] = silhouette_score(Xs, lab)
print(scores)

best_k = max(scores, key=scores.get)
print("Chosen k:", best_k)

plt.plot(list(scores), list(scores.values()), marker="o")
plt.xlabel("k"); plt.ylabel("Silhouette")
plt.tight_layout()
plt.savefig(FIG / "silhouette_scores.png", dpi=160)
plt.close()

{2: np.float64(0.43294018131896606), 3: np.float64(0.33650963406696927), 4: np.float64(0.3372544953400026), 5: np.float64(0.31609724178691406), 6: np.float64(0.31332900440932415), 7: np.float64(0.3100731581775175), 8: np.float64(0.300781560439033)}
Chosen k: 2


In [ ]:
km = KMeans(n_clusters=best_k, random_state=42, n_init=20)
rfm["Cluster"] = km.fit_predict(Xs)

profile = rfm.groupby("Cluster")[features].mean().round(2)
profile["Size"] = rfm.groupby("Cluster").size()
profile.to_csv(ART / "cluster_centroids_business_scale.csv")
print(profile)

whale = int(profile["Monetary"].idxmax())
print("Whale cluster:", whale)

plt.figure(figsize=(7, 5))
for c, g in rfm.groupby("Cluster"):
    plt.scatter(g["Frequency"], g["Monetary"], s=16, alpha=.55, label=f"Cluster {c}")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("Frequency"); plt.ylabel("Monetary"); plt.legend()
plt.tight_layout()
plt.savefig(FIG / "cluster_map.png", dpi=160)
plt.close()

def profile_text(row):
    return f"Recency≈{row.Recency:.0f} days, Frequency≈{row.Frequency:.1f} orders, Monetary≈£{row.Monetary:,.0f}, size={int(row.Size)}"

for idx, row in profile.iterrows():
    print(f"Cluster {idx}: {profile_text(row)}")

         Recency  Frequency  Monetary  Size
Cluster                                    
0         134.14       1.67    497.74  2671
1          25.88       8.44   4548.26  1667
Whale cluster: 1
Cluster 0: Recency≈134 days, Frequency≈1.7 orders, Monetary≈£498, size=2671
Cluster 1: Recency≈26 days, Frequency≈8.4 orders, Monetary≈£4,548, size=1667


In [ ]:
w = profile.loc[whale]
brief = f'''TARGETED MARKETING BRIEF — Cluster {whale}
Audience: highest historical-spending customer segment.
Profile: {profile_text(w)}
Actions:
1. Offer premium bundles or early access instead of blanket discounts.
2. Use loyalty/VIP messaging and personalized cross-sell recommendations.
3. Trigger re-engagement only when recency worsens, to avoid over-messaging valuable customers.
Measurement: track repeat purchase rate, average order value, and campaign margin against a control group.
'''
print(brief)
(ART / "targeted_marketing_brief.txt").write_text(brief)

TARGETED MARKETING BRIEF — Cluster 1
Audience: highest historical-spending customer segment.
Profile: Recency≈26 days, Frequency≈8.4 orders, Monetary≈£4,548, size=1667
Actions:
1. Offer premium bundles or early access instead of blanket discounts.
2. Use loyalty/VIP messaging and personalized cross-sell recommendations.
3. Trigger re-engagement only when recency worsens, to avoid over-messaging valuable customers.
Measurement: track repeat purchase rate, average order value, and campaign margin against a control group.



525